In [1]:
import pyranges
import pandas
import scanpy
import hdf5plugin
import anndata
import cellrank
import matplotlib
from matplotlib import pyplot
import cellrank
import numpy
import gseapy
import pygam
import seaborn
import pydeseq2
import pydeseq2.dds
import pydeseq2.ds
import scipy

In [2]:
# Load data from Cellrank and SDEvelo

working_directory = "RNA Sequencing Data/"
base_name = "cellrank_1.4"
adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} cells.h5ad"
)
dead_end_adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} dead end genes.h5ad"
)
mESC_adata = scanpy.read_h5ad(
    working_directory+f"/{base_name} mESC genes.h5ad"
)
driver_df = pandas.read_csv(
    working_directory+f"{base_name} driver genes.csv",
    index_col=0
)

In [3]:
# Load adata with raw counts and filter

adata_raw = scanpy.read_h5ad(
    working_directory+f"/splice_counts_mm10.h5ad"
)

# Calculate proportion of mitochondrial genes
adata_raw.var["mt"] = adata_raw.var_names.str.startswith("mt-")
scanpy.pp.calculate_qc_metrics(
    adata_raw, qc_vars=["mt"], inplace=True, percent_top=[], log1p=False
)

scanpy.pp.calculate_qc_metrics(
    adata_raw, inplace=True, percent_top=[], log1p=False
)

# Filter cells by counts per cell, total counts and percent mitochondrial
included_cells = (adata_raw.obs["n_genes_by_counts"] >= 200)*(adata_raw.obs['total_counts'] <= 150000)*(adata_raw.obs['pct_counts_mt'] < 10)
adata_raw = adata_raw[included_cells]
print(adata_raw)

View of AnnData object with n_obs × n_vars = 12791 × 48526
    obs: 'barcode', 'batch', 'sample', 'group', 'day', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'ensemble_ids', 'gene_symbol', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'sample_colors'
    obsm: 'X_fdl', 'X_umap'
    layers: 'ambiguous', 'matrix', 'spliced', 'unspliced'


# Differential expression pseudobulk analysis with PyDEseq2

In [97]:
filtered_adata = adata_raw[:, adata.var_names].copy()
filtered_adata.X = filtered_adata.X.toarray()

In [98]:
# Combine counts within samples

sample_names = adata.obs["sample"].unique()

pseudobulk_rows = []
pseudobulk_index = []
sample_metadata_list = []

for sample in sample_names:
    cells_mask = filtered_adata.obs["sample"] == sample
    sample_X = filtered_adata.X[cells_mask]
    summed_counts = sample_X.sum(axis=0)
        
    pseudobulk_rows.append(summed_counts)
    pseudobulk_index.append(sample)
    
    # Determine the group of the sample
    group = adata.obs.loc[cells_mask, "group"].iloc[0]
    day = adata.obs.loc[cells_mask, "day"].iloc[0]
    sample_metadata_list.append({"sample": sample, "group": group, "day": day})

# Create the counts dataframe
counts_df = pandas.DataFrame(
    data=pseudobulk_rows,
    index=pseudobulk_index,
    columns=adata.var_names
)

# Create metadata dataframe
metadata_df = pandas.DataFrame(sample_metadata_list).set_index("sample")

In [100]:
# Differential expression analysis with pydeseq2

reprogramming_adata = adata[adata.obs["group"].isin({"Hic2", "control"})].copy()
inference = pydeseq2.dds.DefaultInference(n_cpus=8)
dds = pydeseq2.dds.DeseqDataSet(
    counts=counts_df,
    metadata=metadata_df,
    design="~group",
    refit_cooks=True,
    inference=inference
)

# Fit model
dds.deseq2()


Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.83 seconds.

Fitting dispersion trend curve...
... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.30 seconds.

Fitting LFCs...
... done in 0.19 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.



In [7]:
stat_results = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "control"], inference=inference)

# compute p-values and adjusted p-values
stat_results.summary()

Log2 fold change & Wald test p-value: group Hic2 vs control
                 baseMean  log2FoldChange     lfcSE      stat    pvalue  \
gene_names                                                                
Rb1cc1        2239.163976       -0.283591  0.244634 -1.159246  0.246356   
Fam150a         33.706046        0.335673  1.152714  0.291203  0.770896   
Prex2          155.234879        1.573466  0.503496  3.125081  0.001778   
Sulf1          677.858231        0.077258  0.591976  0.130508  0.896164   
Msc            423.997047        2.369996  0.716943  3.305699  0.000947   
...                   ...             ...       ...       ...       ...   
Egfl6          166.625885        0.470281  0.781868  0.601483  0.547518   
Tmsb4x      115027.155006        0.654934  0.384154  1.704876  0.088218   
Usp9y           21.430362        2.368054  1.032823  2.292798  0.021860   
Erdr1         1123.037550        0.974630  0.549626  1.773260  0.076186   
AC168977.1       7.812825       -0.62421

Running Wald tests...
... done in 0.11 seconds.



In [8]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
stat_results.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_names,,,,,,
Krt14,1462.170985,-4.686814,0.675246,-6.940899,3.896137e-12,1.262998e-09
Ovol1,340.259652,-3.484112,0.724528,-4.808799,1.518397e-06,7.981844e-05
Tlx2,762.591789,-3.304927,0.750462,-4.403856,1.063436e-05,3.568447e-04
Atf3,489.743202,-2.159268,0.554218,-3.896062,9.776934e-05,2.291101e-03
Hmgb3,1503.378041,-1.429928,0.386190,-3.702652,2.133573e-04,4.153266e-03
Zbtb7c,747.489340,-1.675318,0.482144,-3.474728,5.113722e-04,8.152614e-03
Cdh1,1715.529009,-2.070593,0.713841,-2.900635,3.724076e-03,3.792319e-02
Hic2,1040.204742,1.227000,0.448022,2.738708,6.168113e-03,5.404045e-02
Tcf7l1,1565.285438,1.031554,0.439377,2.347767,1.888635e-02,1.150939e-01


In [142]:
# Most upregulated in control

stat_results.results_df.query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_names,,,,,,
2310046K23Rik,37.150773,-8.101822,1.859812,-4.356259,1.323045e-05,4.361563e-04
Padi4,81.129161,-5.525528,0.887838,-6.223574,4.859564e-10,9.451853e-08
Mal2,397.453478,-5.088056,0.869783,-5.849797,4.921734e-09,7.363671e-07
Vsig8,70.732354,-4.959581,0.871917,-5.688136,1.284334e-08,1.561268e-06
Prss32,1024.286463,-4.922016,0.986613,-4.988802,6.075483e-07,3.938938e-05
Krt14,1462.170985,-4.686814,0.675246,-6.940899,3.896137e-12,1.262998e-09
Ly6g6c,5682.670391,-4.670438,0.962784,-4.850971,1.228587e-06,6.827435e-05
Lgals7,9929.055723,-4.641645,0.854930,-5.429268,5.658576e-08,4.785187e-06
Pkp1,401.631709,-4.622098,0.650179,-7.108967,1.169150e-12,4.547994e-10


In [145]:
# Most upregulated in Hic2

stat_results.results_df.query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
gene_names,,,,,,
Xist,792.024515,3.977165,1.005491,3.955448,7.639151e-05,1.857269e-03
Cd36,25.023779,3.873931,1.143431,3.387989,7.040699e-04,1.029636e-02
Marc1,24.456462,3.576419,0.892136,4.008825,6.102151e-05,1.522021e-03
Stra8,89.375272,3.541480,1.127707,3.140425,1.687032e-03,2.089985e-02
Xirp2,83.190523,3.082889,0.872794,3.532207,4.121060e-04,6.909881e-03
Cdh5,19.222726,3.024164,1.058710,2.856462,4.283913e-03,4.166105e-02
Spink6,436.255228,2.897080,0.983585,2.945431,3.225055e-03,3.465597e-02
Myl3,35.093383,2.895067,0.861486,3.360551,7.778707e-04,1.119690e-02
Bhmt,39.421950,2.885319,1.006768,2.865921,4.157980e-03,4.063955e-02


# Differential expression in bulk data with PyDEseq2

Article describing PyDESeq2: https://academic.oup.com/bioinformatics/article/39/9/btad547/7260507

Article describing DESeq2: https://link.springer.com/article/10.1186/s13059-014-0550-8

In [105]:
bulk_counts = pandas.read_csv("RNA Sequencing Data/Bulk RNA sequencing data Hic2/counts2.tsv", sep="\t", index_col=0).T
samples = bulk_counts.index.to_list()
print(samples)

['Day2_BFP_Rep1', 'Day2_BFP_Rep2', 'Day2_BFP_Rep3', 'Day2_Hic2_Rep1', 'Day2_Hic2_Rep2', 'Day2_Hic2_Rep3', 'Day4_BFP_Rep1', 'Day4_BFP_Rep1-2', 'Day4_BFP_Rep2', 'Day4_BFP_Rep3', 'Day4_Hic2_Rep1', 'Day4_Hic2_Rep1-2', 'Day4_Hic2_Rep2', 'Day4_Hic2_Rep3', 'Day6_BFP_Rep1', 'Day6_BFP_Rep2', 'Day6_BFP_Rep3', 'Day6_Hic2_Rep1', 'Day6_Hic2_Rep2', 'Day6_Hic2_Rep3', 'Day10_BFP_Rep1', 'Day10_BFP_Rep2', 'Day10_BFP_Rep3', 'Day10_Hic2_Rep1', 'Day10_Hic2_Rep2', 'Day10_Hic2_Rep3', 'ESC_BFP_Rep1', 'ESC_BFP_Rep2', 'ESC_BFP_Rep3', 'ESC_Hic2-gRNA_Rep1', 'ESC_Hic2-gRNA_Rep2', 'ESC_Hic2-gRNA_Rep3', 'ESC_Zeo-gRNA_Rep1', 'ESC_Zeo-gRNA_Rep2', 'ESC_Zeo-gRNA_Rep3', 'iPSC_BFP_Rep1', 'iPSC_BFP_Rep2', 'iPSC_BFP_Rep3', 'iPSC_Hic2_Rep1', 'iPSC_Hic2_Rep2', 'iPSC_Hic2_Rep3', 'MEF_BFP_Rep1', 'MEF_BFP_Rep2', 'MEF_BFP_Rep3', 'MEF_Hic2_Rep1', 'MEF_Hic2_Rep2', 'MEF_Hic2_Rep3']


In [106]:
# Create metadata dataframe

sample_metadata_list = []

for sample in samples:
    sample_name_list = sample.split("_")
    if ("Day" in sample_name_list[0]):
        day = int(sample_name_list[0][3:])
        experiment = "reprogramming"
    else:
        day = 0
        experiment = sample_name_list[0]
    group = sample_name_list[1]
    replicate_list = sample_name_list[2][3:].split("-")
    replicate = replicate_list[0]
    if len(replicate_list) == 2:
        batch = int(replicate_list[1])
    else:
        batch = 1 if experiment in {"reprogramming", "MEF"} else 2
    
    sample_metadata_list.append({"sample": sample, "day": day, "experiment": experiment, "group": group, "replicate": replicate, "batch": batch})

# Create metadata dataframe
metadata_df = pandas.DataFrame(sample_metadata_list).set_index("sample")

metadata_df

,day,experiment,group,replicate,batch
sample,,,,,
Day2_BFP_Rep1,2,reprogramming,BFP,1,1
Day2_BFP_Rep2,2,reprogramming,BFP,2,1
Day2_BFP_Rep3,2,reprogramming,BFP,3,1
Day2_Hic2_Rep1,2,reprogramming,Hic2,1,1
Day2_Hic2_Rep2,2,reprogramming,Hic2,2,1
Day2_Hic2_Rep3,2,reprogramming,Hic2,3,1
Day4_BFP_Rep1,4,reprogramming,BFP,1,1
Day4_BFP_Rep1-2,4,reprogramming,BFP,1,2
Day4_BFP_Rep2,4,reprogramming,BFP,2,1


In [107]:
# Fix duplicated columns
duplicated_columns = bulk_counts.columns[bulk_counts.columns.duplicated()].to_list()
print(duplicated_columns)

# These likely result from Excel misinterpreting gene names as dates, possibly MARC1 and MARC2. They can be ignored.
bulk_counts = bulk_counts.drop(columns=duplicated_columns)

['Mar-01', 'Mar-02']


In [108]:
# Find columns that are also in the adata object

variable_genes = adata.var.index.to_numpy()
common_genes = numpy.intersect1d(variable_genes, bulk_counts.columns.to_numpy())

## Differential expression during the reprogramming experiment

In [109]:
reprogramming_samples = metadata_df.query("experiment == \"reprogramming\"").index.to_list()
print(metadata_df.loc[reprogramming_samples])

                  day     experiment group replicate  batch
sample                                                     
Day2_BFP_Rep1       2  reprogramming   BFP         1      1
Day2_BFP_Rep2       2  reprogramming   BFP         2      1
Day2_BFP_Rep3       2  reprogramming   BFP         3      1
Day2_Hic2_Rep1      2  reprogramming  Hic2         1      1
Day2_Hic2_Rep2      2  reprogramming  Hic2         2      1
Day2_Hic2_Rep3      2  reprogramming  Hic2         3      1
Day4_BFP_Rep1       4  reprogramming   BFP         1      1
Day4_BFP_Rep1-2     4  reprogramming   BFP         1      2
Day4_BFP_Rep2       4  reprogramming   BFP         2      1
Day4_BFP_Rep3       4  reprogramming   BFP         3      1
Day4_Hic2_Rep1      4  reprogramming  Hic2         1      1
Day4_Hic2_Rep1-2    4  reprogramming  Hic2         1      2
Day4_Hic2_Rep2      4  reprogramming  Hic2         2      1
Day4_Hic2_Rep3      4  reprogramming  Hic2         3      1
Day6_BFP_Rep1       6  reprogramming   B

In [125]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[reprogramming_samples],
    metadata=metadata_df.loc[reprogramming_samples],
    design="~C(day) + C(batch) + group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
reprogramming_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
reprogramming_de_stats.summary()

Fitting size factors...
... done in 0.04 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.93 seconds.

Fitting dispersion trend curve...
... done in 0.51 seconds.

Fitting MAP dispersions...
... done in 2.11 seconds.

Fitting LFCs...
... done in 2.24 seconds.

Calculating cook's distance...
... done in 0.05 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       18118.223009       -0.237734  0.046419 -5.121532  3.030633e-07   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45        1046.343936        0.800273  0.094942  8.429116  3.482852e-17   
H19          4548.672607        0.729453  0.114614  6.364408  1.960450e-10   
Scml2         183.121673       -0.602971  0.144454 -4.174135  2.991209e-05   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.286036        0.360430  3.104910  0.116084  9.075861e-01   
AC145556.1      0.781459       -1.160610  1.788140 -0.649060  5.162997e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 2.18 seconds.



In [126]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Myc", "Ovol2", "Klf4", "Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
reprogramming_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Hic2,3160.107379,2.198816,0.106077,20.728511,1.916259e-95,2.517219e-92
Zbtb7c,1521.129230,-0.943320,0.058151,-16.221943,3.528526e-59,8.875744e-57
Krt8,21598.666438,-1.363864,0.088721,-15.372461,2.504698e-53,5.018948e-51
Krt14,5110.427482,-4.106556,0.314381,-13.062360,5.402643e-39,5.300643e-37
Nanog,1323.501794,2.370827,0.257670,9.201027,3.545454e-20,9.461880e-19
Zfp42,2688.373406,1.344900,0.149763,8.980156,2.703821e-19,6.666513e-18
Hmgb3,1523.258054,-0.857015,0.103484,-8.281647,1.214838e-16,2.360299e-15
Sall4,1673.703697,0.972394,0.122353,7.947441,1.904039e-15,3.218084e-14
Ovol1,680.945017,-1.451013,0.185601,-7.817914,5.370601e-15,8.644510e-14
Tcf7l1,582.207950,1.263931,0.164272,7.694150,1.424378e-14,2.177080e-13


In [127]:
# Higher expression in control

reprogramming_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
2310046K23Rik,121.261593,-5.026695,0.526483,-9.547682,1.326310e-21,4.142749e-20
Krt14,5110.427482,-4.106556,0.314381,-13.062360,5.402643e-39,5.300643e-37
Krt6a,8104.897532,-3.767656,0.231002,-16.310079,8.368502e-60,2.198591e-57
Mlc1,6.222436,-3.765394,0.772700,-4.873032,1.098985e-06,6.230041e-06
Slurp1,817.514135,-3.661155,0.243806,-15.016651,5.712186e-51,9.931224e-49
Sel1l3,35.552444,-3.592158,0.570257,-6.299195,2.991958e-10,2.773220e-09
Chit1,745.571491,-3.592043,0.181281,-19.814787,2.219452e-87,2.099158e-84
Mmp13,74.221433,-3.585201,0.291299,-12.307646,8.239182e-35,6.537431e-33
Krt17,5304.148465,-3.505668,0.151732,-23.104406,4.181106e-118,1.098469e-114
Fam180a,50.590588,-3.467625,0.341159,-10.164249,2.863197e-24,1.124589e-22


The high upregulation of 2310046K23Rik, also known as Small proline rich protein 5 (Sprr5) is very interesting.

Function of human Sprr5 according to Uniprot: "Positively regulates keratinocyte differentiation by inducing genes associated with epidermal differentiation".

In [128]:
# Higher expression with Hic2

reprogramming_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Ifi27l2a,60.427621,4.964874,0.849305,5.845811,5.041075e-09,3.957378e-08
Eras,129.003245,4.766440,0.745645,6.392374,1.633294e-10,1.556959e-09
3830417A13Rik,274.669327,4.546757,0.306071,14.855229,6.434730e-50,1.071473e-47
Oasl2,329.182784,4.419285,0.438522,10.077691,6.933599e-24,2.609246e-22
Pramef12,591.836100,4.304483,0.708941,6.071709,1.265565e-09,1.073508e-08
Dnmt3l,2023.817274,4.146114,0.878535,4.719352,2.365970e-06,1.268270e-05
Nr0b1,399.205895,4.031501,0.463751,8.693253,3.522062e-18,7.916269e-17
Napsa,4.379697,4.014911,1.029909,3.898317,9.686348e-05,3.921139e-04
Cdh5,11.450055,3.893308,0.550767,7.068885,1.561839e-12,1.900653e-11
Gm13119,27.830769,3.853006,0.458546,8.402669,4.364452e-17,8.934847e-16


## Differential expression in day 2

In [ ]:
day_2_samples = metadata_df.query("experiment == \"reprogramming\" and day == 2").index.to_list()
print(metadata_df.loc[day_2_samples])

                day     experiment group replicate  batch
sample                                                   
Day2_BFP_Rep1     2  reprogramming   BFP         1      1
Day2_BFP_Rep2     2  reprogramming   BFP         2      1
Day2_BFP_Rep3     2  reprogramming   BFP         3      1
Day2_Hic2_Rep1    2  reprogramming  Hic2         1      1
Day2_Hic2_Rep2    2  reprogramming  Hic2         2      1
Day2_Hic2_Rep3    2  reprogramming  Hic2         3      1


In [121]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_2_samples],
    metadata=metadata_df.loc[day_2_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_2_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_2_de_stats.summary()

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.44 seconds.

Fitting dispersion trend curve...
... done in 0.45 seconds.

Fitting MAP dispersions...
... done in 1.43 seconds.

Fitting LFCs...
... done in 1.33 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       22128.672245       -0.120252  0.037410 -3.214455  1.306925e-03   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45         690.247113        0.478694  0.089751  5.333594  9.628743e-08   
H19          2598.156315        0.973740  0.318548  3.056811  2.237051e-03   
Scml2         286.322931       -0.440445  0.153644 -2.866664  4.148229e-03   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.000000             NaN       NaN       NaN           NaN   
AC145556.1      1.836549       -1.380029  2.047491 -0.674010  5.003051e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 2.10 seconds.



In [122]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
day_2_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Hic2,3149.191898,2.688885,0.076548,35.126718,2.635957e-270,4.470846e-266
Krt8,6886.109100,-1.620890,0.046294,-35.013120,1.420759e-268,1.204875e-264
Krt14,3064.054179,-2.215632,0.087545,-25.308377,2.583242e-141,3.983124e-138
Cdh1,1811.915900,-1.431488,0.080460,-17.791400,8.239497e-71,3.176139e-68
Zbtb7c,1545.528898,-0.915841,0.073941,-12.386172,3.105169e-35,2.831547e-33
Krtdap,638.725020,1.533055,0.138921,11.035443,2.577781e-28,1.607417e-26
Tlx2,536.024936,-1.211808,0.131036,-9.247933,2.288763e-20,8.782740e-19
Ovol1,331.654674,-1.328884,0.146448,-9.074094,1.146274e-19,4.084445e-18
Hmgb3,1901.452329,-0.379427,0.067962,-5.582933,2.364953e-08,2.756836e-07
Tcf7l1,469.488292,0.578413,0.116473,4.966068,6.832397e-07,6.384809e-06


In [123]:
# Higher expression in control

day_2_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Cbr2,9.181110,-2.816140,0.937512,-3.003844,2.665916e-03,1.138670e-02
Krt17,1025.852472,-2.815688,0.103277,-27.263386,1.153250e-163,2.794324e-160
Slurp1,241.005104,-2.786119,0.181877,-15.318704,5.734927e-53,1.157977e-50
Fermt3,167.084349,-2.715851,0.236266,-11.494872,1.399881e-30,9.651778e-29
Fam180a,76.626846,-2.688414,0.304611,-8.825742,1.087367e-18,3.591692e-17
Pkp1,1929.939402,-2.591549,0.180137,-14.386529,6.287301e-47,9.437071e-45
Gpx2,25.385435,-2.501531,0.523653,-4.777073,1.778649e-06,1.542314e-05
Ly6g6c,56.906671,-2.430245,0.375380,-6.474090,9.538479e-11,1.489707e-09
Mapk13,240.980212,-2.396511,0.186562,-12.845643,9.099282e-38,9.297164e-36
S100a14,19.860539,-2.368664,0.559988,-4.229849,2.338487e-05,1.645771e-04


In [124]:
# Higher expression with Hic2

day_2_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Oasl2,257.102054,6.886760,1.053968,6.534127,6.398168e-11,1.019307e-09
Ifi44,136.271695,6.128665,1.019703,6.010248,1.852398e-09,2.511473e-08
Nr0b1,23.558192,6.090100,1.121658,5.429550,5.649644e-08,6.283516e-07
Ifi27l2a,11.812878,5.933982,1.613442,3.677840,2.352170e-04,1.325861e-03
Ifit3b,61.723969,5.706857,1.554940,3.670145,2.424129e-04,1.362521e-03
Rsad2,214.173270,5.387152,1.107387,4.864742,1.146061e-06,1.028484e-05
Ifit3,118.796074,5.177538,0.979090,5.288111,1.235858e-07,1.316670e-06
Ifit1,260.327155,4.863718,0.857393,5.672680,1.405806e-08,1.693457e-07
Rtp4,62.698740,4.843640,1.201816,4.030269,5.571313e-05,3.616343e-04
Usp18,120.299860,4.758689,0.881617,5.397683,6.750696e-08,7.444639e-07


## Differential expression in day 4

In [116]:
day_4_samples = metadata_df.query("experiment == \"reprogramming\" and day == 4").index.to_list()
print(metadata_df.loc[day_4_samples])

                  day     experiment group replicate  batch
sample                                                     
Day4_BFP_Rep1       4  reprogramming   BFP         1      1
Day4_BFP_Rep1-2     4  reprogramming   BFP         1      2
Day4_BFP_Rep2       4  reprogramming   BFP         2      1
Day4_BFP_Rep3       4  reprogramming   BFP         3      1
Day4_Hic2_Rep1      4  reprogramming  Hic2         1      1
Day4_Hic2_Rep1-2    4  reprogramming  Hic2         1      2
Day4_Hic2_Rep2      4  reprogramming  Hic2         2      1
Day4_Hic2_Rep3      4  reprogramming  Hic2         3      1


In [117]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_4_samples],
    metadata=metadata_df.loc[day_4_samples],
    design="~C(day) + C(batch) + group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_4_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_4_de_stats.summary()

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.37 seconds.

Fitting dispersion trend curve...
... done in 0.48 seconds.

Fitting MAP dispersions...
... done in 1.56 seconds.

Fitting LFCs...
... done in 1.61 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       15061.034487       -0.145678  0.048888 -2.979812  2.884256e-03   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45         695.773380        0.949563  0.116376  8.159410  3.366663e-16   
H19          3270.342112        0.798852  0.092818  8.606611  7.525230e-18   
Scml2         239.054249       -0.541207  0.135172 -4.003823  6.232701e-05   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.863819        1.292333  2.205131  0.586057  5.578371e-01   
AC145556.1      0.339300        0.601040  3.320829  0.180991  8.563748e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 1.96 seconds.



In [118]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
day_4_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Hic2,3154.780976,2.399564,0.065990,36.362376,1.675002e-289,9.103215e-286
Krtdap,1793.064199,2.325247,0.065549,35.473522,1.258797e-275,4.560833e-272
Krt8,24075.847629,-1.650879,0.059111,-27.928501,1.202923e-171,8.716779e-169
Zbtb7c,1758.713380,-1.095422,0.053464,-20.488793,2.710318e-93,3.954336e-91
Krt14,7186.538884,-4.235427,0.293805,-14.415787,4.117190e-47,2.067058e-45
Zfp42,427.157105,1.555553,0.137043,11.350841,7.345297e-30,1.878581e-28
Sall4,1312.697523,1.117913,0.104806,10.666470,1.460712e-26,3.147117e-25
Nanog,53.434685,3.237392,0.343363,9.428489,4.160391e-21,6.582441e-20
Ovol1,796.688716,-0.757347,0.083485,-9.071703,1.171713e-19,1.685762e-18
Cdh1,10584.047067,-0.565965,0.065088,-8.695453,3.454493e-18,4.417484e-17


In [119]:
# Higher expression in control

day_4_de_stats.results_df.query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Zxda,19.182632,-7.144257,2.270178,-3.147003,1.649531e-03,4.716447e-03
Gm5940,5.245388,-5.785067,2.061074,-2.806822,5.003286e-03,1.304172e-02
Gm35507,8.275602,-5.754740,1.447637,-3.975264,7.030127e-05,2.516929e-04
Ly6d,5.064573,-5.744730,1.859937,-3.088669,2.010550e-03,5.663040e-03
Gm44196,4.808731,-5.710786,1.883808,-3.031511,2.433331e-03,6.735189e-03
Lce3b,8.120451,-5.659103,1.449224,-3.904919,9.425678e-05,3.305983e-04
Krt90,350.081052,-5.639549,0.414021,-13.621420,2.987037e-42,1.278252e-40
Myoz2,24.224179,-5.569297,0.840379,-6.627125,3.422872e-11,2.430899e-10
Wnt7a,211.924221,-5.426897,0.483151,-11.232305,2.829978e-29,7.014925e-28
2310046K23Rik,143.752868,-5.419234,0.696467,-7.781036,7.193321e-15,7.082229e-14


In [120]:
# Higher expression with Hic2

day_4_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Gm13119,4.560295,5.567292,1.796201,3.099482,1.938593e-03,5.482405e-03
Eras,11.339190,5.439741,1.109805,4.901528,9.509425e-07,4.331142e-06
Napsa,3.667996,5.312775,1.870530,2.840251,4.507802e-03,1.185952e-02
Oc90,5.969510,5.227858,1.475699,3.542632,3.961549e-04,1.264798e-03
Dnmt3l,5.432173,5.067096,1.517523,3.339058,8.406302e-04,2.533545e-03
Nr0b1,55.178169,5.060812,0.465392,10.874295,1.528328e-27,3.468092e-26
Rhox6,11.462765,4.912741,1.007439,4.876465,1.080041e-06,4.882305e-06
Pramef12,11.219521,4.909294,1.002603,4.896551,9.753340e-07,4.432947e-06
Gulo,3.260914,4.356487,1.595638,2.730248,6.328675e-03,1.616296e-02
Cdh5,3.118053,4.239004,1.556186,2.723970,6.450235e-03,1.645408e-02


## Differential expression in day 6

In [80]:
day_6_samples = metadata_df.query("experiment == \"reprogramming\" and day == 6").index.to_list()
print(metadata_df.loc[day_6_samples])

                day     experiment group replicate  batch
sample                                                   
Day6_BFP_Rep1     6  reprogramming   BFP         1      1
Day6_BFP_Rep2     6  reprogramming   BFP         2      1
Day6_BFP_Rep3     6  reprogramming   BFP         3      1
Day6_Hic2_Rep1    6  reprogramming  Hic2         1      1
Day6_Hic2_Rep2    6  reprogramming  Hic2         2      1
Day6_Hic2_Rep3    6  reprogramming  Hic2         3      1


In [129]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_6_samples],
    metadata=metadata_df.loc[day_6_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_6_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_6_de_stats.summary()

Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.36 seconds.

Fitting dispersion trend curve...
... done in 0.44 seconds.

Fitting MAP dispersions...
... done in 1.32 seconds.

Fitting LFCs...
... done in 1.36 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
Gnai3       22363.371052       -0.307583  0.093112 -3.303380  9.552682e-04   
Pbsn            0.000000             NaN       NaN       NaN           NaN   
Cdc45        1137.191588        1.182272  0.128137  9.226605  2.793445e-20   
H19          3896.764956        0.465219  0.336784  1.381358  1.671688e-01   
Scml2          82.045704       -1.696386  0.424089 -4.000073  6.332304e-05   
...                  ...             ...       ...       ...           ...   
BX571804.1      0.000000             NaN       NaN       NaN           NaN   
AC154773.1      0.000000             NaN       NaN       NaN           NaN   
AL662853.1      0.000000             NaN       NaN       NaN           NaN   
AC145556.1      1.191800       -3.722858  4.235882 -0.878886  3.794630e-01   
CT868734.1      0.000000             NaN       NaN       NaN           NaN   

       

... done in 2.01 seconds.



In [130]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
day_6_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Hic2,2797.623961,2.157327,0.111354,19.373565,1.290083e-83,9.388194e-81
Tfcp2l1,2117.184424,2.836544,0.194685,14.569883,4.366368e-48,9.687480e-46
Krt14,7410.892477,-4.181132,0.287351,-14.550630,5.786792e-48,1.268423e-45
Krtdap,3475.638326,1.373563,0.111199,12.352334,4.731854e-35,5.093883e-33
Hmgb3,1381.829875,-1.399331,0.131310,-10.656667,1.623102e-26,1.028888e-24
Nanog,540.294456,2.516980,0.237262,10.608442,2.722494e-26,1.707943e-24
Zfp42,1517.458376,1.986789,0.195344,10.170728,2.678989e-24,1.441978e-22
Tcf7l1,432.475319,2.128565,0.227501,9.356281,8.259261e-21,3.560681e-19
Sall4,1691.440867,1.498192,0.177879,8.422521,3.684691e-17,1.155786e-15
Krt8,35915.605695,-0.866945,0.105122,-8.247045,1.623600e-16,4.748898e-15


In [131]:
# Higher expression in control

day_6_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Gfap,47.669095,-8.078114,1.545994,-5.225190,1.739768e-07,1.698048e-06
Sel1l3,23.755821,-8.039324,2.174451,-3.697174,2.180125e-04,1.170346e-03
Col2a1,45.522950,-8.011511,1.505257,-5.322354,1.024332e-07,1.036467e-06
Lat2,12.345648,-7.095768,2.233821,-3.176516,1.490554e-03,6.430555e-03
Il6,11.332284,-6.972185,2.257527,-3.088416,2.012263e-03,8.352523e-03
2310046K23Rik,241.807171,-6.371918,1.463529,-4.353804,1.337953e-05,9.430989e-05
Cd53,15.917120,-5.545834,1.424303,-3.893717,9.871964e-05,5.714306e-04
Lypd2,40.918428,-5.138319,0.860197,-5.973420,2.323303e-09,3.049628e-08
Mmp13,117.183501,-4.779892,0.492072,-9.713805,2.633189e-22,1.247542e-20
Fam180a,58.557126,-4.468817,0.763884,-5.850122,4.912113e-09,6.154688e-08


In [132]:
# Higher expression with Hic2

day_6_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Dnmt3l,818.910694,10.672582,0.858088,12.437625,1.632871e-35,1.800413e-33
Dppa3,133.792752,10.501275,1.984438,5.291814,1.211091e-07,1.212624e-06
Pramef12,477.612385,10.484930,1.044874,10.034637,1.073530e-23,5.580209e-22
Eras,87.377929,8.923089,1.450481,6.151812,7.660256e-10,1.086358e-08
Rhox9,284.444808,8.418665,0.697033,12.077853,1.382826e-33,1.413357e-31
Robo4,12.799660,7.117509,2.226828,3.196254,1.392245e-03,6.069760e-03
Wfdc2,64.778082,7.015705,0.976164,7.187018,6.622165e-13,1.359786e-11
Pla2g10,7.546254,6.354666,2.621542,2.424018,1.534985e-02,4.698970e-02
Bhmt,78.678302,6.268316,0.775500,8.082934,6.322708e-16,1.761547e-14
Pramel6,6.254222,6.085642,2.510027,2.424532,1.532813e-02,4.693901e-02


## Day 10 differential expression

In [134]:
day_10_samples = metadata_df.query("experiment == \"reprogramming\" and day == 10").index.to_list()
print(metadata_df.loc[day_10_samples])

                 day     experiment group replicate  batch
sample                                                    
Day10_BFP_Rep1    10  reprogramming   BFP         1      1
Day10_BFP_Rep2    10  reprogramming   BFP         2      1
Day10_BFP_Rep3    10  reprogramming   BFP         3      1
Day10_Hic2_Rep1   10  reprogramming  Hic2         1      1
Day10_Hic2_Rep2   10  reprogramming  Hic2         2      1
Day10_Hic2_Rep3   10  reprogramming  Hic2         3      1


In [135]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=16)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[day_10_samples],
    metadata=metadata_df.loc[day_10_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
day_10_de_stats = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
day_10_de_stats.summary()

Fitting size factors...
... done in 0.02 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 1.38 seconds.

Fitting dispersion trend curve...
... done in 0.45 seconds.

Fitting MAP dispersions...
... done in 1.43 seconds.

Fitting LFCs...
... done in 1.35 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Gnai3       14599.801326       -0.486742  0.110552 -4.402836  0.000011   
Pbsn            0.000000             NaN       NaN       NaN       NaN   
Cdc45        1823.385673        0.472664  0.184033  2.568364  0.010218   
H19          9032.655059        0.583415  0.142092  4.105908  0.000040   
Scml2         103.792330        0.022729  0.348827  0.065159  0.948047   
...                  ...             ...       ...       ...       ...   
BX571804.1      0.000000             NaN       NaN       NaN       NaN   
AC154773.1      0.000000             NaN       NaN       NaN       NaN   
AL662853.1      0.000000             NaN       NaN       NaN       NaN   
AC145556.1      0.000000             NaN       NaN       NaN       NaN   
CT868734.1      0.000000             NaN       NaN       NaN       NaN   

                padj  
Gnai3       0.000073  
Pbsn     

... done in 1.97 seconds.



In [136]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
day_10_de_stats.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Ovol1,523.721234,-3.065469,0.268917,-11.399329,4.213554e-30,6.988580e-28
Hic2,3528.417087,1.412919,0.134361,10.515811,7.304973e-26,7.556882e-24
Hmgb3,1472.708056,-1.320122,0.129233,-10.215083,1.697342e-24,1.507255e-22
Krt8,18156.640526,-1.300506,0.149296,-8.710930,3.013925e-18,1.447045e-16
Hoxa7,313.478151,-2.416212,0.312123,-7.741205,9.847861e-15,3.224090e-13
Krt14,1833.668617,-5.782565,0.751295,-7.696798,1.395177e-14,4.423350e-13
Tsc22d1,5885.198478,0.899993,0.124330,7.238765,4.527873e-13,1.159055e-11
Nanog,5271.214218,1.179392,0.166227,7.095067,1.292888e-12,3.081587e-11
Tlx2,203.466775,-2.257631,0.325591,-6.933944,4.092656e-12,8.966760e-11
Zic2,702.258569,1.868672,0.282547,6.613673,3.748997e-11,6.985944e-10


In [138]:
# Higher expression in control

day_10_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Lilrb4a,9.670870,-6.725212,2.376187,-2.830254,4.651113e-03,1.596429e-02
Bfsp2,8.464746,-6.536837,2.364571,-2.764492,5.701150e-03,1.894625e-02
Krt14,1833.668617,-5.782565,0.751295,-7.696798,1.395177e-14,4.423350e-13
2310046K23Rik,66.749600,-5.765423,0.748905,-7.698470,1.377052e-14,4.380693e-13
Lypd2,30.611673,-5.379791,1.034593,-5.199910,1.993845e-07,1.934001e-06
Acod1,365.572994,-5.123413,0.789755,-6.487343,8.736343e-11,1.521959e-09
Mlc1,16.067620,-4.955775,1.282753,-3.863389,1.118248e-04,6.026350e-04
Slurp1,495.962366,-4.856608,0.969936,-5.007144,5.524358e-07,4.958334e-06
Krt6a,5384.824223,-4.785563,0.453454,-10.553586,4.889477e-26,5.137535e-24
Gfap,61.583979,-4.594034,0.575792,-7.978631,1.479650e-15,5.331256e-14


In [139]:
# Higher expression with Hic2

day_10_de_stats.results_df.loc[common_genes].query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Tex101,64.224640,6.985983,0.949437,7.358024,1.866531e-13,5.014648e-12
Hand1,695.537998,6.115260,0.258542,23.652838,1.103560e-123,2.214734e-119
Ifi27l2a,232.672652,5.887109,1.396644,4.215182,2.495765e-05,1.562108e-04
1700013H16Rik,74.436598,5.801094,1.615614,3.590644,3.298625e-04,1.574693e-03
Gm5091,35.882148,5.670047,0.998624,5.677859,1.363907e-08,1.634164e-07
Cdh5,25.377575,5.192595,1.089449,4.766259,1.876779e-06,1.518753e-05
Cadps,21.525663,4.874195,1.148286,4.244758,2.188291e-05,1.391093e-04
Xirp2,433.072464,4.812337,0.306422,15.704918,1.399590e-55,1.404419e-52
Ifi44,149.483631,4.798597,0.441547,10.867697,1.642963e-27,2.047989e-25
Stra8,196.754285,4.269137,0.410677,10.395368,2.602796e-25,2.548074e-23


## Differential expression in MEFs

In [64]:
mef_samples = metadata_df.query("experiment == \"MEF\"").index.to_list()
print(metadata_df.loc[mef_samples])

               day experiment group replicate  batch
sample                                              
MEF_BFP_Rep1     0        MEF   BFP         1      1
MEF_BFP_Rep2     0        MEF   BFP         2      1
MEF_BFP_Rep3     0        MEF   BFP         3      1
MEF_Hic2_Rep1    0        MEF  Hic2         1      1
MEF_Hic2_Rep2    0        MEF  Hic2         2      1
MEF_Hic2_Rep3    0        MEF  Hic2         3      1


In [65]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=8)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[mef_samples, common_genes],
    metadata=metadata_df.loc[mef_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
stat_results = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2", "BFP"], inference=inference)
stat_results.summary()

Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.20 seconds.

Fitting dispersion trend curve...
... done in 0.04 seconds.

Fitting MAP dispersions...
... done in 0.24 seconds.

Fitting LFCs...
... done in 0.14 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2 vs BFP
                  baseMean  log2FoldChange     lfcSE      stat        pvalue  \
0610040J01Rik    21.265536       -0.403483  0.497757 -0.810603  4.175938e-01   
1110008L16Rik   324.582825        0.239663  0.203499  1.177710  2.389124e-01   
1110038B12Rik  1234.185766        0.028475  0.202006  0.140961  8.879011e-01   
1500009L16Rik   289.683772        0.891146  0.162259  5.492105  3.971718e-08   
1500015O10Rik    71.716345       -1.477486  0.287123 -5.145826  2.663468e-07   
...                    ...             ...       ...       ...           ...   
Zmiz1          6521.568472        0.028579  0.138333  0.206597  8.363244e-01   
Zp3               0.667739       -2.910684  3.596741 -0.809256  4.183680e-01   
Zscan4a           0.000000             NaN       NaN       NaN           NaN   
Zscan4c           0.000000             NaN       NaN       NaN           NaN   
Zscan4e           0.000000             NaN       NaN       NaN  

... done in 0.11 seconds.



In [66]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
stat_results.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Hic2,3534.500069,4.469488,0.174119,25.669161,2.583999e-145,4.509078e-142
Krtdap,57.269042,2.305351,0.457112,5.043291,4.575935e-07,7.677891e-06
Zic2,314.603117,0.660156,0.176438,3.741579,1.828678e-04,1.477334e-03
Tcf7l2,1461.158516,0.411018,0.150840,2.724854,6.432983e-03,2.679130e-02
Hoxa7,806.661901,0.347961,0.135321,2.571375,1.012956e-02,3.859406e-02
Tet1,275.564610,0.472414,0.190623,2.478264,1.320234e-02,4.689391e-02
Zbtb7c,124.670887,-0.638847,0.262957,-2.429475,1.512072e-02,5.183822e-02
Klf2,196.442771,-0.859362,0.378279,-2.271767,2.310061e-02,7.276276e-02
Nanog,12.499910,1.211845,0.582080,2.081924,3.734944e-02,1.045203e-01
Sall4,3.086663,2.999819,1.444811,2.076271,3.786889e-02,1.053927e-01


In [67]:
# Higher expression in control

stat_results.results_df.query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Chac1,86.435946,-3.177753,0.301851,-10.527550,6.449088e-26,1.125366e-23
Lyz2,19.130831,-2.740876,0.714613,-3.835471,1.253237e-04,1.061601e-03
F5,363.014411,-2.648490,0.453023,-5.846260,5.027480e-09,1.438189e-07
S100a4,4789.558842,-2.458065,0.155282,-15.829698,1.941495e-56,1.693954e-53
Tnmd,20.900485,-2.438697,0.721062,-3.382090,7.193650e-04,4.446867e-03
Lgals7,13.455553,-2.269549,0.680326,-3.335971,8.500207e-04,5.079747e-03
Gdf15,83.102744,-2.232503,0.425322,-5.248976,1.529467e-07,3.103395e-06
Sncg,1073.423977,-2.214171,0.262866,-8.423207,3.663174e-17,3.760140e-15
Mmp13,412.721768,-2.211962,0.236938,-9.335597,1.004238e-20,1.347996e-18
9530053A07Rik,41.139792,-2.187889,0.402724,-5.432725,5.549982e-08,1.181063e-06


In [68]:
# Higher expression with Hic2

stat_results.results_df.query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Ifi27l2a,44.558107,5.176916,0.949596,5.451705,4.988906e-08,1.088205e-06
Hic2,3534.500069,4.469488,0.174119,25.669161,2.583999e-145,4.509078e-142
Isg15,1792.130563,4.261802,0.774550,5.502295,3.748793e-08,8.840059e-07
Oasl2,1080.804964,4.112931,1.144824,3.592631,3.273561e-04,2.370276e-03
Ifi44,1395.079071,4.012075,0.730660,5.491031,3.995942e-08,9.174892e-07
Ap3b2,13.942344,3.912476,1.024189,3.820071,1.334132e-04,1.119260e-03
Rtp4,527.356070,3.723732,0.612262,6.081921,1.187508e-09,4.228984e-08
1600025M17Rik,4.677879,3.687203,1.371406,2.688630,7.174598e-03,2.918339e-02
Nts,8.900478,3.560124,1.001310,3.555464,3.773119e-04,2.644214e-03
Usp18,679.254187,3.545879,0.778479,4.554884,5.241458e-06,6.575572e-05


## Differential expression in ESCs

In [75]:
esc_samples = metadata_df.query("experiment == \"ESC\"").index.to_list()
print(metadata_df.loc[esc_samples])

                    day experiment      group replicate  batch
sample                                                        
ESC_BFP_Rep1          0        ESC        BFP         1      2
ESC_BFP_Rep2          0        ESC        BFP         2      2
ESC_BFP_Rep3          0        ESC        BFP         3      2
ESC_Hic2-gRNA_Rep1    0        ESC  Hic2-gRNA         1      2
ESC_Hic2-gRNA_Rep2    0        ESC  Hic2-gRNA         2      2
ESC_Hic2-gRNA_Rep3    0        ESC  Hic2-gRNA         3      2
ESC_Zeo-gRNA_Rep1     0        ESC   Zeo-gRNA         1      2
ESC_Zeo-gRNA_Rep2     0        ESC   Zeo-gRNA         2      2
ESC_Zeo-gRNA_Rep3     0        ESC   Zeo-gRNA         3      2


In [76]:
# Fit differential expression model

inference = pydeseq2.dds.DefaultInference(n_cpus=8)
dds = pydeseq2.dds.DeseqDataSet(
    counts=bulk_counts.loc[esc_samples, common_genes],
    metadata=metadata_df.loc[esc_samples],
    design="~group",
    refit_cooks=True,
    inference=inference
)

dds.deseq2()

# Calculate statistics
stat_results = pydeseq2.ds.DeseqStats(dds, contrast=["group", "Hic2-gRNA", "Zeo-gRNA"], inference=inference)
stat_results.summary()

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.20 seconds.

Fitting dispersion trend curve...
... done in 0.04 seconds.

Fitting MAP dispersions...
... done in 0.24 seconds.

Fitting LFCs...
... done in 0.17 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: group Hic2-gRNA vs Zeo-gRNA
                  baseMean  log2FoldChange     lfcSE      stat        pvalue  \
0610040J01Rik    51.921452        0.527384  0.290160  1.817561  6.913131e-02   
1110008L16Rik   722.601741       -0.083498  0.148153 -0.563595  5.730299e-01   
1110038B12Rik  1389.865872       -0.755943  0.127443 -5.931615  2.999698e-09   
1500009L16Rik   154.560178       -0.125743  0.267273 -0.470467  6.380211e-01   
1500015O10Rik     5.207505        3.085208  1.198319  2.574613  1.003523e-02   
...                    ...             ...       ...       ...           ...   
Zmiz1           785.722730        1.124409  0.246163  4.567739  4.930129e-06   
Zp3              97.979012       -0.822176  0.307450 -2.674180  7.491220e-03   
Zscan4a         177.497754        0.543751  0.570026  0.953905  3.401316e-01   
Zscan4c         205.143733        0.600216  0.857092  0.700293  4.837443e-01   
Zscan4e         210.310155        0.010496  0.606083  

... done in 0.11 seconds.



In [77]:
# Whether genes of interest are up- or downregulated

genes_of_interest = ["Krtdap", "Krt8", "Krt14", "Nanog", "Tet1", "Cdh1", "Ovol1", "Atf3", "Hic2", "Sall4", "Shisa8", 'Hmgb3', 'Hoxa7', 'Tlx2', 'Zbtb7c', 'Klf2', 'Mycn', 'Rest', 'Tcf7l1', 'Tcf7l2', 'Tfcp2l1', 'Tsc22d1',
       'Zfp42', 'Zic2']
stat_results.results_df.loc[genes_of_interest].sort_values("padj")

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Nanog,4052.544730,-2.439410,0.090728,-26.886934,3.122445e-159,2.693109e-156
Tfcp2l1,7033.005586,-2.041081,0.153019,-13.338782,1.376903e-40,7.917192e-39
Tet1,12266.356463,-1.048578,0.088047,-11.909249,1.059267e-32,4.060525e-31
Zfp42,12778.669838,-1.464963,0.128270,-11.420961,3.285796e-30,1.111372e-28
Atf3,46.400934,2.926515,0.336155,8.705852,3.151986e-18,4.647159e-17
Rest,6352.040976,-0.464189,0.076798,-6.044300,1.500605e-09,1.023140e-08
Zic2,1306.400655,1.908681,0.318025,6.001669,1.952993e-09,1.305780e-08
Mycn,4152.026270,1.605242,0.297809,5.390168,7.039168e-08,3.891848e-07
Hoxa7,13.261125,-2.789823,0.685144,-4.071877,4.663576e-05,1.665563e-04
Hmgb3,626.869728,0.439919,0.110975,3.964125,7.366551e-05,2.536387e-04


In [78]:
# Higher expression in control

stat_results.results_df.query("padj < 0.05 and log2FoldChange < 0").sort_values("log2FoldChange").head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Aox3,15.310374,-7.671577,1.993699,-3.847911,1.191294e-04,3.944303e-04
Nostrin,7.526519,-5.966891,2.024740,-2.946991,3.208825e-03,8.022062e-03
Hand1,6.241400,-5.819437,2.181096,-2.668125,7.627591e-03,1.745039e-02
Msc,323.344069,-5.801899,0.366322,-15.838242,1.694922e-56,2.088386e-54
Nkain4,22.767862,-4.461031,0.727754,-6.129863,8.795458e-10,6.133297e-09
Col3a1,2.609835,-4.351009,1.902813,-2.286619,2.221807e-02,4.498377e-02
Calml4,25.678331,-4.303625,0.804977,-5.346268,8.978636e-08,4.901313e-07
Spink1,28.530931,-4.115843,1.006824,-4.087946,4.352089e-05,1.577175e-04
Nanos3,18.706729,-3.985595,0.659684,-6.041673,1.525243e-09,1.035844e-08
Slc6a1,133.897434,-3.946398,0.349372,-11.295696,1.378086e-29,4.402219e-28


Interestingly, Myc expression is much lower in control cells

In [79]:
# Higher expression with Hic2

stat_results.results_df.query("padj < 0.05 and log2FoldChange > 0").sort_values("log2FoldChange", ascending=False).head(50)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
Ms4a4d,20.326466,4.825755,1.246411,3.871721,1.080695e-04,3.612788e-04
Tmem176b,3.612990,4.643747,1.610812,2.882861,3.940807e-03,9.534211e-03
Myl3,2.556719,4.126678,1.713033,2.408989,1.599680e-02,3.381676e-02
Lbp,8.501897,4.060805,0.942797,4.307191,1.653408e-05,6.409277e-05
Igfbp5,6.627575,3.910211,1.232140,3.173512,1.506068e-03,4.040385e-03
Mfap2,9.475300,3.895051,0.923540,4.217523,2.470002e-05,9.323313e-05
4930550L24Rik,3.824585,3.843686,1.625766,2.364231,1.806754e-02,3.763326e-02
Enpp2,73.021343,3.801862,0.648664,5.861068,4.598995e-09,2.938247e-08
Fas,12.493819,3.782708,0.793289,4.768384,1.857093e-06,8.452467e-06
Megf10,6.832699,3.764085,1.113708,3.379777,7.254465e-04,2.059033e-03


# Differential expression with SCVI-tools